# STEP 6 — 1단계 고치기 (백본 × 증강 2×2)

## 왜 지금 이걸 하나

STEP 5 에서 **holdout 을 처음 열었더니 1단계만 무너졌습니다.**

| | val | **holdout** | |
|---|---:|---:|---|
| 1단계 AUROC | 0.8143 | **0.7412** | 📉 −0.073 |
| 2단계 macro-F1 | 0.5313 | **0.5582** | 📈 +0.027 |

같은 분할·같은 규칙인데 **한쪽만** 이러면 데이터가 아니라 모델 문제입니다.
그리고 그게 파이프라인 전체를 깎습니다 — 2단계만 0.5582 인데 최종은 0.4432.
**1단계가 0.115 를 깎고 있습니다.** 스크리닝 recall 도 0.9303 으로 목표(0.95) 미달입니다.

## 용의자 두 명

**① 화질 지름길** — 크롭 감사 실측:

| | 선명도 중앙값 | 흐린 사진 |
|---|---:|---:|
| A7 무증상(정상) | **50** | 57.2% |
| 병변 6종 | 274 | 23~50% |

**정상 사진이 계통적으로 흐립니다.** 아픈 데는 신경 써서 가까이 또렷하게 찍고
멀쩡한 데는 대충 찍기 때문으로 보입니다. 모델이 "흐리면 정상" 을 배웠다면,
개체가 바뀌어 촬영 습관이 달라지는 순간 무너집니다.

그리고 그 신호는 **배포에 존재하지 않습니다** — 보호자는 아픈지 모르는 채로 찍으니까요.

**② 표현력·사전학습** — `resnet50` 은 2015년 모델이고 ImageNet-1k 로만 배웠습니다.
더 큰 사전학습(in21k)을 거친 백본이면 분포가 바뀌어도 덜 무너질 수 있습니다.

## 2×2 로 한 번에 가릅니다

| | `default` 증강 | `photometric` 증강 |
|---|---|---|
| **`resnet50`** | 기준선 재측정 | 증강 축 |
| **`effnetv2_s`** | 백본 축 | 둘 다 |

### 왜 `photometric` 을 1단계에는 다시 시도하나

2단계에서는 **실패**했습니다 (macro-F1 −0.03). 이유는 2단계가 병변의 **종류**를
구분해야 하는데 그 단서가 **질감**이라, 흐리게 하면 답이 지워지기 때문입니다.

**1단계는 다릅니다.** 유무만 보면 되니 질감 디테일이 덜 중요하고,
반대로 화질 지름길은 **실측으로 존재**합니다. 같은 약이 다른 환자에게는 들을 수 있습니다.

### 왜 `convnextv2_base` 가 아니라 `effnetv2_s` 인가

* 원하는 레버는 "**더 큰 사전학습**" 인데 `effnetv2_s` 도 in21k 입니다 (약 21M).
  `convnextv2_base` 는 거기에 "모델 4배 키우기"(약 89M)가 얹힌 별개 질문입니다
* 여기서 **증강이 표적 치료, 백본은 보조 베팅**입니다. 보조에 시간을 제일 많이 쓸 이유가 없습니다
* `effnetv2_s` 는 **모바일 배포 1순위 후보**라 이기면 그대로 씁니다.
  89M 짜리는 이겨도 폰에 못 넣습니다

`convnextv2_base` 는 04(2단계 백본 비교)에서 봅니다 — 거기선 정확도 상한이 목적이라 맞습니다.

## 판정 기준 — **실험 전에** 못 박습니다 (규칙 2)

규칙은 `src/experiments.py` 의 `stage1_report()` 에 있습니다 (노트북 셀이 아니라 — 규칙 3).

```
잡음 폭   AUROC ±0.01   (같은 설정 두 실행: 0.8192 / 0.8155)
          흐림 하락 ±5%p (STEP 5 에서 ±3%p → ±5%p 상향)

photometric 채택  ← AUROC 를 0.01 이상 안 깎으면서 흐림 하락이 5%p 이상 감소
백본 교체 채택    ← AUROC 가 0.01 이상 상승
둘 다 아니면      → 그 축은 닫음
```

## ⚠️ holdout 은 여기서 안 봅니다

4개 중에 고르는 데 holdout 을 쓰면 **그 순간 오염됩니다** — 더 이상 "처음 보는
데이터" 가 아니게 됩니다. 고른 조합을 **풀 데이터로 다시 학습한 뒤에만** 엽니다.

그래서 여기서 쓰는 지표는 두 개입니다:
* **val AUROC** — 후보를 고르는 기준
* **흐림 교란 하락** — 지름길이 막혔는지 보는 기준 (val 점수만으로는 구분이 안 됩니다)

## 설정

1단계만 / `full` 크롭 / 384px / 학습셋 55% / 12에폭.
⚠️ 서브셋·짧은 에폭이라 **절대값은 풀 학습과 다릅니다.** 후보를 줄이는 용도입니다
(STEP 4B 에서 확인 — 순위는 옮겨가고 절대값은 안 옮겨갑니다).

In [ ]:
# ── 0. 환경 준비 (Colab / Kaggle 공통) ──────────────────────────
# 이 셀 하나가 리포 동기화 → 패키지 설치 → 환경 감지까지 다 합니다.
# 리포를 직접 다운로드하거나 드라이브에 올릴 필요 없습니다.
# 다시 실행하면 항상 최신 코드로 맞춰집니다 (로컬 수정은 덮어씁니다).
import os, sys, subprocess

REPO   = "https://github.com/gayeoniee/deeplearning_test.git"
NAME   = "deeplearning_test"
# ⚠️ 브랜치를 "main" 으로 **못 박으면 안 됩니다.** 아래 reset --hard 가
#    작업 브랜치를 통째로 덮어써서, 방금 만든 코드가 사라진 채로 몇 시간을
#    돌게 됩니다. 이미 리포 안에서 돌고 있으면 **지금 브랜치를 그대로 씁니다.**
#    바꾸려면 환경변수:  export DOG_SKIN_BRANCH=main
BRANCH = os.environ.get("DOG_SKIN_BRANCH", "")
_cwd   = os.getcwd()
if os.path.basename(_cwd) == NAME and os.path.isdir(os.path.join(_cwd, ".git")):
    DIR = _cwd            # 이미 리포 안에서 재실행 중 (중첩 clone 방지)
    if not BRANCH:
        BRANCH = subprocess.run(["git", "-C", DIR, "rev-parse", "--abbrev-ref", "HEAD"],
                                capture_output=True, text=True).stdout.strip() or "main"
else:
    # ⚠️ Kaggle 을 먼저 봅니다. Kaggle 이미지에도 /content 가 있어서
    #    /content 를 먼저 보면 Kaggle 세션인데 /content 에 clone 합니다.
    BASE = ("/kaggle/working" if os.path.isdir("/kaggle/working")
            else "/content" if os.path.isdir("/content") else _cwd)
    DIR = os.path.join(BASE, NAME)

BRANCH = BRANCH or "main"
if os.path.isdir(os.path.join(DIR, ".git")):
    # 이미 받아둔 경우: 최신으로 강제 동기화 (shallow clone 에서도 안전)
    subprocess.run(["git", "-C", DIR, "fetch", "--depth", "1", "origin", BRANCH], check=False)
    subprocess.run(["git", "-C", DIR, "reset", "--hard", f"origin/{BRANCH}"], check=False)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO, DIR], check=True)

os.chdir(DIR)
if DIR not in sys.path:
    sys.path.insert(0, DIR)

# ⚠️ 중요: 이미 import 된 src.* 는 파이썬이 캐시하고 있어서
#    파일을 갱신해도 옛날 코드가 그대로 쓰입니다. 캐시를 비웁니다.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("작업 디렉터리:", os.getcwd())
print("코드 버전   :", subprocess.run(["git", "-C", DIR, "log", "--oneline", "-1"],
                                      capture_output=True, text=True).stdout.strip())

# 패키지 설치는 **uv 로 통일**합니다 (pip 보다 훨씬 빠릅니다).
# ⚠️ Colab/Kaggle 이미지에는 uv 가 없어서, uv 자체만 pip 로 한 번 받습니다.
#    --system = 가상환경을 새로 만들지 않고 이미 있는 파이썬에 그대로 설치.
#    (torch/numpy/pandas 는 이미 깔려 있으므로 여기서 안 건드립니다)
# albumentations 는 import 할 때마다 PyPI 에 버전 확인 요청을 보냅니다.
# Kaggle 은 외부 네트워크가 막혀 있어 타임아웃(2초)만 기다리다 끝납니다 — 꺼둡니다.
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

_PKGS = ["timm", "imagehash", "pyarrow", "grad-cam", "albumentations"]

# ⚠️ 일부 이미지(런팟 PyTorch 등)는 파이썬이 **externally managed** 라
#    (PEP 668) --system 설치를 거부합니다. Colab/Kaggle 에는 없는 문제라
#    처음엔 안 넣었다가 런팟에서 첫 셀이 바로 죽었습니다.
#    --break-system-packages 를 붙여 한 번 더 시도합니다.
def _install(args: list[str]) -> bool:
    return subprocess.run(args, check=False).returncode == 0


_ok = False
if _install([sys.executable, "-m", "pip", "install", "-q", "uv"]):
    _base = [sys.executable, "-m", "uv", "pip", "install", "-q", "--system"]
    _ok = _install(_base + _PKGS)
    if not _ok:
        _ok = _install(_base + ["--break-system-packages"] + _PKGS)
if not _ok:
    print("[env] uv 로 설치하지 못해 pip 으로 대체합니다")
    _p = [sys.executable, "-m", "pip", "install", "-q"]
    if not _install(_p + _PKGS):
        _install(_p + ["--break-system-packages"] + _PKGS)

# 한글 그래프 폰트 (Colab 기본에는 한글이 없어 □ 로 나옵니다)
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
try:
    import matplotlib.pyplot as plt, matplotlib.font_manager as fm
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

MY_NOTEBOOK_VERSION = "2026-08-25.1"   # ★ 이 셀(=이 .ipynb)의 버전

from src import env
from src.config import CFG, CLASSES, CLASS_KO
E = env.describe()
env.set_seed(42)

# 환경 판정이 이상하면(예: Kaggle 인데 colab 이라고 나오면) 근거를 봅니다
if E.env != "local":
    env.diagnose()

# ⚠️ 노트북 셀은 git pull 로 갱신되지 않습니다 (src/ 만 최신이 됩니다).
#    낡은 .ipynb 를 몇 시간 돌리고 나서 알게 되면 늦으므로 지금 확인합니다.
from src.config import NOTEBOOK_VERSION as _repo_nb
if MY_NOTEBOOK_VERSION != _repo_nb:
    print("\n" + "!" * 62)
    print(f"⚠️ 이 노트북이 낡았습니다 — 내 셀 {MY_NOTEBOOK_VERSION} / 리포 {_repo_nb}")
    print("   src/ 는 최신이지만 **셀 내용은 예전 것**입니다.")
    print("   GitHub 에서 notebooks/*.ipynb 를 다시 받아 Import 하세요:")
    print("   Kaggle → File → Import Notebook / Colab → 파일 → 노트 업로드")
    print("!" * 62 + "\n")
else:
    print(f"[nb] 노트북 최신 ({_repo_nb})")


---
## 1. 데이터 붙이기 + 사전 검증

In [ ]:
# Drive 마운트는 **진짜 Colab VM** 에서만 시도합니다.
# ⚠️ Kaggle 에도 google.colab 패키지와 /content 가 있어서, 환경 판정을 잘못하면
#    Kaggle 에서 drive.mount() 를 부르고 NotImplementedError 로 죽습니다.
if env.can_mount_drive():
    env.mount_drive()
else:
    print(f"[env] {E.env} — Drive 마운트 없이 진행합니다")

# 전처리 결과를 붙입니다. 두 가지 형태를 다 받습니다:
#   · Colab  : Drive 의 dogskin_prepared.zip → 로컬 디스크로 해제
#   · Kaggle : /kaggle/input/<데이터셋>/crops,manifests → 링크만 연결
#              (Kaggle 은 업로드한 zip 을 알아서 풀어둡니다. 복사하면 20GB 제한에 걸려요)
env.load_prepared()          # 경로를 직접 주려면: env.load_prepared("/kaggle/input/dogskin-prepared")

# ── 다른 환경에서 학습한 체크포인트 가져오기 (Colab → Kaggle 이주) ──────
#    Colab 에서 이미 학습을 끝냈다면, Drive 의 dogskin_work/checkpoints 를
#    Kaggle 데이터셋으로 올린 뒤 그 경로를 여기에 주세요.
#    가져온 실험은 '완료' 로 인식되어 학습 셀이 ⏭️ 로 건너뜁니다.
#
# train.import_checkpoints("/kaggle/input/dogskin-ckpt")

# 세션이 끊겨도 남는 저장소 확인
_persist = env.persist_root()
if _persist is None:
    print("\n🚨 세션 밖 저장소가 없습니다 — 지금 학습하면 끊길 때 체크포인트가 사라집니다.")
    print("   위 셀에서 Drive 마운트가 됐는지 확인하세요 (env.mount_drive()).")
else:
    print(f"\n✅ 중단 대비 저장소: {_persist}")
    if E.env == "kaggle":
        print("   ⚠️ Kaggle 은 세션이 끝나면 /kaggle/working 이 사라질 수 있습니다.")
        print("      · 짧게 확인만 할 때  : 그냥 진행 (세션 안에서는 이어받기가 됩니다)")
        print("      · 긴 학습을 돌릴 때  : 우측 상단 [Save Version] →")
        print("                             'Save & Run All (Commit)' 로 돌리세요.")
        print("                             브라우저를 닫아도 끝까지 돌고, 출력이 보존됩니다.")
        print("      · 설정에 Persistence 항목이 보이면 'Files' 로 켜두면 더 안전합니다")
    else:
        print("   매 에폭 체크포인트를 여기로 복사합니다. 세션이 끊기면 노트북을 처음부터")
        print("   다시 돌리세요 — 끝난 학습은 건너뛰고 끊긴 학습만 이어서 합니다.")

In [ ]:
import torch
from src import (labels, split, crop, data, models, train, evaluate,
                 stages, experiments, robust)
from src.config import CLASSES_STAGE1

env.require_gpu()
DEV = "cuda" if torch.cuda.is_available() else "cpu"

IMG_SIZE    = 384          # 03 해상도 실험에서 채택
EPOCHS      = 12           # 서브셋 스윕용 (03b 와 동일)
SUBSET      = 0.55         # 학습셋만 줄입니다. 검증셋은 그대로
STAGE1_CROP = "full"       # ⚠️ 1단계는 ROI 크롭을 안 씁니다 (crop.choose_stage1_tag 참고)
MODELS      = ("resnet50", "effnetv2_s")
AUGS        = ("default", "photometric")

# 실측 기준 (STEP 4D / STEP 5). 이번 resnet50/default 가 여기서 크게 벗어나면
# 데이터·환경이 달라진 것이므로 먼저 원인을 찾으세요.
# ⚠️ 아래는 **풀 데이터 25에폭** 값이고 이번은 서브셋 12에폭이라 더 낮게 나옵니다.
BASE_FULL = {"val_auroc": 0.8155, "holdout_auroc": 0.7412,
             "screening_recall": 0.9303, "precision": 0.650}

df = labels.load(env.work_root()/"manifests"/"manifest_final.parquet")

# ── 학습 전에 전부 확인합니다 ────────────────────────────────────
# 03c 에서 배운 것: 크롭 확인을 학습 루프 안에 두면 78분 뒤에 터집니다.
have = crop.available_tags()
print(f"사용 가능한 크롭 태그: {have}")
if STAGE1_CROP not in have:
    raise SystemExit(
        f"❌ 크롭 '{STAGE1_CROP}' 이 없습니다. 붙어 있는 것: {have}\n"
        f"   Kaggle 우측 [Add Input] 에서 dogskin-full 을 붙이세요.")

d = crop.switch_tag(df, STAGE1_CROP)      # 커버리지 95% 미만이면 여기서 멈춥니다
view = stages.to_stage1(d)
split.verify(view, fold=0, strict=True)   # 누수가 있으면 여기서 에러
tr, va = split.get_fold(view, 0)

N_TRAIN = int(len(tr) * SUBSET)
print(f"\n1단계 뷰 {len(view):,}행  ·  train {len(tr):,} → {N_TRAIN:,}({SUBSET:.0%})"
      f"  ·  val {len(va):,}")
print(f"조건 {len(MODELS)} × {len(AUGS)} = {len(MODELS) * len(AUGS)}개")

---
## 2. 시작 전에 — 몇 시간 걸릴지 먼저 잽니다

합성 텐서로 GPU 속도만 재므로 **백본당 20초** 안쪽입니다. 학습은 아직 시작 안 합니다.

> 이번 프로젝트에서 "몇 시간 걸릴지 모르고 돌렸다가 뒤통수" 를 여러 번 맞았습니다.
> 여기서 총 예상 시간을 보고 **너무 길면 그만두거나 서브셋을 줄이세요.**

⚠️ GPU 속도만 잰 **하한**입니다. 데이터 로딩이 병목이면 실제는 더 걸립니다
(실측: 384px 에서 GPU 112 img/s 상한, 로더 90 img/s).

In [ ]:
est = experiments.estimate_runtime(
    list(MODELS), img_size=IMG_SIZE, n_train=N_TRAIN, epochs=EPOCHS,
    n_conditions=len(MODELS) * len(AUGS))

# VRAM 이 빠듯하면 배치가 자동으로 줄어듭니다. 14.6GB(T4) 를 넘으면 OOM 위험.
for _r in est["rows"]:
    if (_r.get("peak_vram_gb") or 0) > 13.0:
        print(f"🚨 {_r['model']} 이 VRAM {_r['peak_vram_gb']}GB 를 씁니다 — OOM 위험.")
        print("   config 의 batch_size 를 직접 낮추거나 grad_accum 을 올리세요.")

---
## 3. 2×2 학습

⚠️ 여기서부터 오래 걸립니다. 위 예상 시간을 보고 진행하세요.
세션이 끊겨도 `train.fit(resume=True)` 가 마지막 에폭부터 이어받습니다.

In [ ]:
# 위에서 이미 검증한 view 를 씁니다 (여기서 크롭·분할로 실패할 일이 없어야 합니다)
runs = []
for _m in MODELS:
    for _a in AUGS:
        runs.append(experiments.train_and_measure(
            view, stage=1, img_size=IMG_SIZE, crop_tag=STAGE1_CROP,
            device=DEV, epochs=EPOCHS, model_name=_m, aug=_a,
            subset_frac=SUBSET,
            # 배율 교란은 1단계에서 이미 9.1% 로 통과했습니다 (STEP 4A).
            # 지금 궁금한 건 **화질** 이므로 그쪽만 잽니다 — 시간도 아낍니다.
            measure_robust=False, measure_blur=True, n_robust=2000))

---
## 4. 판정

기준은 `experiments.stage1_report()` 에 있습니다 — **실험 전에** 정해뒀고
노트북 셀이 아니라 `src/` 에 있어서 `git pull` 로 갱신됩니다 (규칙 2·3).

In [ ]:
verdict = experiments.stage1_report(runs, base_model="resnet50", base_aug="default")

import json
W = env.work_root(); (W/"reports").mkdir(parents=True, exist_ok=True)
keep = ("stage", "model_name", "aug", "img_size", "crop_tag", "exp_name",
        "epochs", "batch_size", "minutes", "best_epoch", "n_epochs", "converged",
        "subset_frac", "n_train", "auroc", "threshold", "precision",
        "blur_drop", "blur_worst", "blur_worst_at")
(W/"reports"/"step6_stage1.json").write_text(json.dumps({
    "img_size": IMG_SIZE, "epochs": EPOCHS, "subset_frac": SUBSET,
    "crop": STAGE1_CROP, "baseline_full_run": BASE_FULL,
    "noise": {"auroc": experiments.AUROC_NOISE, "blur_pp": experiments.BLUR_NOISE_PP},
    "runs": [{k: r[k] for k in keep if k in r} for r in runs],
    "verdict": verdict,
}, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\n저장: {W/'reports'/'step6_stage1.json'}")

# 다음 노트북(풀 학습)이 쓸 수 있게 꾸러미로 내보냅니다
train.export_release(
    exps=[r["exp_name"] for r in runs],
    meta={"실험": "STEP 6 1단계 2×2", "크롭": STAGE1_CROP, "입력": f"{IMG_SIZE}px",
          "서브셋": f"{SUBSET:.0%}", "에폭": EPOCHS,
          "고른 조합": str(verdict.get("best", "판정 없음"))},
    files={"reports/step6_stage1.json": json.loads(
        (W/"reports"/"step6_stage1.json").read_text(encoding="utf-8"))},
)

---
## 5. 다음

**이 노트북은 후보를 고르는 것까지입니다.** 서브셋 55% · 12에폭이라 절대값은
풀 학습과 다릅니다.

| 결과 | 다음 |
|---|---|
| 어느 축이든 채택됨 | 그 조합으로 **03 을 풀 데이터 재실행** → holdout 을 다시 엽니다 |
| 둘 다 잡음 안 | 두 축 모두 닫고 `f320` 재크롭(구도 불일치 가설)으로 |

⚠️ **holdout 은 아직 안 봤습니다.** 풀 학습 뒤에 한 번만 엽니다.
지금 holdout 을 보고 조합을 고르면 그 숫자는 더 이상 정직하지 않습니다.

⚠️ Kaggle 이면 **[Save Version] → Save & Run All (Commit)** 으로 돌리고,
끝나면 `release` 폴더를 **New Dataset (Private)** 으로 만들어 두세요.
`READ_ME_FIRST.txt` 에 어떤 실험인지 적혀 있습니다.